In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, max, min, round, rank,
    date_format, weekofyear, month, year, lag,
    when, countDistinct, datediff, to_date
)
from pyspark.sql.window import Window

transactions = spark.table("my_catalog.silver.transactions")

In [0]:
fraud_by_day = (
    transactions
    .groupBy("day_of_week")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .orderBy(col("fraud_count").desc())
)
display(fraud_by_day)

day_of_week,total_transactions,fraud_count,fraud_rate_pct
Sunday,1899044,2646,0.14
Friday,1895372,2284,0.12
Thursday,1918666,2082,0.11
Tuesday,1897678,2037,0.11
Monday,1896914,1747,0.09
Saturday,1902370,1434,0.08
Wednesday,1895871,1102,0.06


In [0]:
fraud_trend = (
    transactions
    .groupBy("transaction_date")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .orderBy("transaction_date")
)
display(fraud_trend)

transaction_date,total_transactions,fraud_count,fraud_rate_pct
2010-01-01,3463,1,0.03
2010-01-02,2989,0,0.0
2010-01-03,3311,1,0.03
2010-01-04,3244,2,0.06
2010-01-05,3330,1,0.03
2010-01-06,3365,0,0.0
2010-01-07,3346,2,0.06
2010-01-08,3016,4,0.13
2010-01-09,3102,1,0.03
2010-01-10,3416,5,0.15


In [0]:
top_fraud_users = (
    transactions
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(
        count("*").alias("fraud_transaction_count"),
        sum("amount").alias("total_fraud_amount"),
        round(avg("amount"), 2).alias("avg_fraud_amount")
    )
    .orderBy(col("fraud_transaction_count").desc())
)
display(top_fraud_users)

client_id,fraud_transaction_count,total_fraud_amount,avg_fraud_amount
1102,58,7615.21,131.3
209,52,4855.87,93.38
27,45,2932.16,65.16
155,44,3536.5000000000005,80.38
1128,43,4699.15,109.28
989,42,7499.97,178.57
1741,42,3575.7200000000003,85.14
1851,42,6102.569999999999,145.3
1649,41,3585.2900000000004,87.45
408,39,3034.9,77.82


In [0]:
weekly_avg = (
    transactions
    .withColumn("week", weekofyear("transaction_date"))
    .withColumn("year", year("transaction_date"))
    .groupBy("client_id", "year", "week")
    .agg(round(avg("amount"), 2).alias("weekly_avg_amount"))
)

user_window = Window.partitionBy("client_id", "year").orderBy("week")

amount_spike = (
    weekly_avg
    .withColumn("prev_week_avg", lag("weekly_avg_amount", 1).over(user_window))
    .withColumn("safe_prev", when(col("prev_week_avg") == 0, None).otherwise(col("prev_week_avg")))
    .withColumn("pct_change", round(
        (col("weekly_avg_amount") - col("prev_week_avg")) / col("safe_prev") * 100, 2
    ))
    .drop("safe_prev")
    .filter(col("pct_change") > 50)
    .orderBy(col("pct_change").desc())
)
display(amount_spike)

client_id,year,week,weekly_avg_amount,prev_week_avg,pct_change
734,2012,52,29.15,0.01,291400.0
1267,2011,43,54.19,0.02,270850.0
1669,2012,15,119.22,0.08,148925.0
1091,2014,27,109.36,0.08,136600.0
1737,2019,36,25.85,0.02,129150.0
1288,2018,46,60.39,0.08,75387.5
1666,2013,10,20.53,0.03,68333.33
465,2013,48,54.14,0.09,60055.56
1406,2011,32,23.84,0.04,59500.0
772,2017,47,17.56,0.03,58433.33


In [0]:
fraud_by_mcc = (
    transactions
    .groupBy("merchant_category")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct"),
        round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("total_fraud_amount")
    )
    .orderBy(col("fraud_rate_pct").desc())
)
display(fraud_by_mcc)

merchant_category,total_transactions,fraud_count,fraud_rate_pct,total_fraud_amount
Cruise Lines,428,165,38.55,185946.78
Music Stores - Musical Instruments,319,76,23.82,7562.26
Miscellaneous Fabricated Metal Products,351,29,8.26,8543.36
"Computers, Computer Peripheral Equipment",2793,204,7.3,25615.11
Floor Covering Stores,334,23,6.89,6643.92
Electronics Stores,6997,402,5.75,61171.38
Miscellaneous Metal Fabrication,391,22,5.63,6943.78
Fabricated Structural Metal Products,408,22,5.39,6234.74
Precious Stones and Metals,5180,242,4.67,26575.96
Coated and Laminated Products,381,17,4.46,5642.57


In [0]:
fraud_by_merchant = (
    transactions
    .filter(col("is_fraud") == 1)
    .groupBy("merchant_id", "merchant_city", "merchant_state")
    .agg(
        count("*").alias("fraud_count"),
        round(sum("amount"), 2).alias("total_fraud_amount")
    )
    .orderBy(col("fraud_count").desc())
    .limit(20)
)
display(fraud_by_merchant)

merchant_id,merchant_city,merchant_state,fraud_count,total_fraud_amount
60569,ONLINE,null,759,82309.67
27092,ONLINE,null,715,64747.02
76639,ONLINE,null,284,43344.88
32858,ONLINE,null,283,27824.99
83018,Rome,Italy,236,13363.74
48919,Rome,Italy,221,18811.81
99370,Rome,Italy,195,12686.28
34490,ONLINE,null,177,23317.75
47399,ONLINE,null,176,8749.92
88260,Rome,Italy,149,5140.36


In [0]:
fraud_by_time = (
    transactions
    .groupBy("time_of_day")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .orderBy(col("fraud_count").desc())
)
display(fraud_by_time)

time_of_day,total_transactions,fraud_count,fraud_rate_pct
morning,5415684,5741,0.11
afternoon,4464677,5693,0.13
evening,1835859,1480,0.08
night,1589695,418,0.03


In [0]:
avg_amount_by_fraud = (
    transactions
    .groupBy("is_fraud")
    .agg(
        count("*").alias("transaction_count"),
        round(avg("amount"), 2).alias("avg_amount"),
        round(min("amount"), 2).alias("min_amount"),
        round(max("amount"), 2).alias("max_amount")
    )
)
display(avg_amount_by_fraud)

is_fraud,transaction_count,avg_amount,min_amount,max_amount
0,13292583,42.91,-500.0,6820.2
1,13332,110.23,-500.0,4978.45


In [0]:
fraud_amount_by_mcc = (
    transactions
    .filter(col("is_fraud") == 1)
    .groupBy("merchant_category")
    .agg(
        round(sum("amount"), 2).alias("total_fraud_amount"),
        count("*").alias("fraud_count")
    )
    .orderBy(col("total_fraud_amount").desc())
)
display(fraud_amount_by_mcc)

merchant_category,total_fraud_amount,fraud_count
Department Stores,225647.19,2251
Cruise Lines,185946.78,165
Wholesale Clubs,113827.65,991
Discount Stores,81214.89,859
Money Transfer,66101.52,725
Electronics Stores,61171.38,402
"Furniture, Home Furnishings, and Equipment Stores",56989.45,170
Miscellaneous Home Furnishing Stores,34238.45,313
Telecommunication Services,33625.04,162
Family Clothing Stores,30051.56,385


In [0]:
daily_losses = (
    transactions
    .filter(col("is_fraud") == 1)
    .filter((col("is_fraud") == 1) & (col("amount") > 0))
    .groupBy("transaction_date")
    .agg(
        round(sum("amount"), 2).alias("total_fraud_loss"),
        count("*").alias("fraud_count")
    )
    .orderBy("transaction_date")
)
display(daily_losses)

transaction_date,total_fraud_loss,fraud_count
2010-01-01,0.19,1
2010-01-03,339.0,1
2010-01-04,11.64,2
2010-01-05,8.76,1
2010-01-07,290.54,1
2010-01-08,383.24,4
2010-01-09,23.1,1
2010-01-10,530.01,5
2010-01-11,302.7,2
2010-01-12,1014.41,1


In [0]:
unique_fraud_users_weekly = (
    transactions
    .filter(col("is_fraud") == 1)
    .withColumn("week", weekofyear("transaction_date"))
    .withColumn("year", year("transaction_date"))
    .groupBy("year", "week")
    .agg(countDistinct("client_id").alias("unique_fraud_users"))
    .orderBy("year", "week")
)
display(unique_fraud_users_weekly)

year,week,unique_fraud_users
2010,1,5
2010,2,5
2010,3,7
2010,4,17
2010,5,16
2010,6,12
2010,7,13
2010,8,13
2010,9,10
2010,10,6


In [0]:
monthly_fraud = (
    transactions
    .withColumn("year_month", date_format("transaction_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct"),
        round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("total_fraud_amount")
    )
    .orderBy("year_month")
)
display(monthly_fraud)

year_month,total_transactions,fraud_count,fraud_rate_pct,total_fraud_amount
2010-01,101209,107,0.11,12253.57
2010-02,93470,259,0.28,32488.11
2010-03,103345,261,0.25,37454.2
2010-04,100169,237,0.24,29130.42
2010-05,104773,274,0.26,27258.86
2010-06,102677,182,0.18,15926.29
2010-07,106034,244,0.23,28099.77
2010-08,107547,229,0.21,24878.75
2010-09,103902,193,0.19,22063.84
2010-10,106150,224,0.21,29450.98


In [0]:
first_fraud = (
    transactions
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(min("transaction_date").alias("first_fraud_date"))
)

behavior_shift = (
    transactions
    .join(first_fraud, on="client_id", how="inner")
    .withColumn("period", when(
        col("transaction_date") < col("first_fraud_date"), "before_fraud"
    ).otherwise("after_fraud"))
    .groupBy("client_id", "period")
    .agg(
        count("*").alias("transaction_count"),
        round(avg("amount"), 2).alias("avg_amount"),
        round(sum("amount"), 2).alias("total_amount")
    )
    .orderBy("client_id", "period")
)
display(behavior_shift)

client_id,period,transaction_count,avg_amount,total_amount
0,after_fraud,5247,49.54,259942.87
0,before_fraud,7548,48.47,365856.8
1,after_fraud,2824,35.91,101404.24
1,before_fraud,7249,32.39,234783.13
100,after_fraud,4380,57.75,252924.35
100,before_fraud,2690,57.18,153824.01
1002,after_fraud,3549,53.55,190031.68
1002,before_fraud,175,52.31,9153.42
1003,after_fraud,9712,92.6,899370.15
1003,before_fraud,1118,94.45,105600.52


In [0]:
fraud_by_value = (
    transactions
    .withColumn("amount_bucket",
        when(col("amount") < 10, "under $10")
        .when((col("amount") >= 10) & (col("amount") < 50), "$10–$50")
        .when((col("amount") >= 50) & (col("amount") < 100), "$50–$100")
        .when((col("amount") >= 100) & (col("amount") < 500), "$100–$500")
        .otherwise("$500+")
    )
    .groupBy("amount_bucket")
    .agg(
        count("*").alias("total_transactions"),
        sum("is_fraud").alias("fraud_count"),
        round(sum("is_fraud") / count("*") * 100, 2).alias("fraud_rate_pct")
    )
    .orderBy(col("fraud_rate_pct").desc())
)
display(fraud_by_value)

amount_bucket,total_transactions,fraud_count,fraud_rate_pct
$500+,43364,356,0.82
$100–$500,1516248,4726,0.31
$50–$100,2899476,2819,0.1
under $10,3574928,2617,0.07
$10–$50,5271899,2814,0.05


In [0]:
fraud_by_day.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_by_day")
fraud_trend.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_trend")
top_fraud_users.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.top_fraud_users")
amount_spike.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.amount_spikes")
fraud_by_mcc.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_by_mcc")
fraud_by_merchant.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_by_merchant")
fraud_by_time.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_by_time")
avg_amount_by_fraud.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.avg_amount_fraud_vs_non")
fraud_amount_by_mcc.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_amount_by_mcc")
daily_losses.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.daily_fraud_losses")
unique_fraud_users_weekly.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.unique_fraud_users_weekly")
monthly_fraud.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.monthly_fraud")
behavior_shift.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.behavior_shift")
fraud_by_value.write.format("delta").mode("overwrite").saveAsTable("my_catalog.gold.fraud_by_value")